# Dynamic Pollutant Processing - Daily/Monthly/Yearly

This notebook processes pollutant data with settings from **config.py**.

**To change settings:**
1. Edit `config.py` to change dates, sample period, paths, etc.
2. Run this notebook

**No need to edit this notebook!**

In [1]:
# Import packages
import osgeo
import xarray as xr
import geopandas as gpd
import rioxarray as rxr
import odc.geo.xr
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load configuration
%run 00_config.ipynb
print_config()

CURRENT CONFIGURATION

Date Range:
  Start: 2024-01-01
  End: 2024-12-31

Processing Settings:
  Sample Period: 1H
  Statistical Method: mean
  Region: All
  Pollutants: SO2, NO2, CO, O3, PM2P5, PM10

Output Path:
  C:\projects\aqi_scad\03_Output_Data\hourly_1h_data


In [4]:
# Load helper functions
%run functions.ipynb

In [5]:
# Load region boundaries
print("Loading region boundaries...")
gdf = gpd.read_file(PATHS['boundary_vector']).to_crs(CONFIG['target_crs'])

# Select region
if CONFIG['region'] == 'All':
    aoi = gdf
else:
    aoi = gdf[gdf[CONFIG['region_field']] == CONFIG['region']]

print(f"Region: {CONFIG['region']}")
print(f"Number of polygons: {len(aoi)}")

Loading region boundaries...
Region: All
Number of polygons: 3


In [6]:
# Get output path based on sample period
output_folder = get_output_path(CONFIG['sample_period'])
print(f"\nOutput folder: {output_folder}")


Output folder: C:\projects\aqi_scad\03_Output_Data\hourly_1h_data


## Process All Pollutants

In [7]:
# Process all pollutants
print("\n" + "="*70)
print("PROCESSING POLLUTANTS")
print("="*70)

for pollutant in CONFIG['pollutant_names']:
    print(f"\n{'='*70}")
    print(f"Processing: {pollutant}")
    print(f"{'='*70}")

    # Load data
    print(f"\n1. Loading {pollutant} data...")
    data_array = load_data(data_path=PATHS['input'], pollutant=pollutant, region=aoi)
    print(f"   Loaded {len(data_array.time)} timesteps")

    # Resample temporally
    print(f"\n2. Resampling to {CONFIG['sample_period']} using {CONFIG['stat_method']}...")
    data_array_temporal = resample_data(
        data_array=data_array,
        start_date=CONFIG['start_date'],
        end_date=CONFIG['end_date'],
        sample_period=CONFIG['sample_period'],
        stat_method=CONFIG['stat_method']
    )
    data_array_temporal = data_array_temporal.compute()
    print(f"   Resampled to {len(data_array_temporal.time)} timesteps")

    # Clip to AOI
    print(f"\n3. Clipping to region...")
    data_array_temporal = data_array_temporal.rio.clip(
        aoi.geometry.values, aoi.crs, all_touched=True, drop=False
    )
    data_array_temporal = data_array_temporal.fillna(PROCESSING['nodata_value'])

    # Export
    print(f"\n4. Exporting TIFFs...")
    identifiers = [i for i in range(len(data_array_temporal.time))]
    export_data_tif(
        data_array_temporal,
        output_folder,
        identifiers,
        pollutant,
        CONFIG['sample_period'],
        CONFIG['stat_method']
    )
    print(f"   Exported {len(identifiers)} TIFF files")
    print(f"\n✓ {pollutant} processing complete!")

print("\n" + "="*70)
print("ALL POLLUTANTS PROCESSED SUCCESSFULLY!")
print("="*70)
print(f"\nOutput location: {output_folder}")
print(f"Total files exported: {len(CONFIG['pollutant_names']) * len(identifiers)}")


PROCESSING POLLUTANTS

Processing: SO2

1. Loading SO2 data...
Loading: C:\projects\aqi_scad\02_Input\CAMS_SO2_ugm3.zarr
  - RAW data range: 0.73 to 79.36
  - Time range: 2024-01-01T00:00:00.000000000 to 2024-12-31T21:00:00.000000000
  - Valid positive values: 410906736
Loading: C:\projects\aqi_scad\02_Input\MOD_SO2_ugm3.zarr
  - RAW data range: -203.55 to 423.85
  - Time range: 2024-01-01T00:00:00.000000000 to 2024-12-30T00:00:00.000000000
  - Valid positive values: 36659481
Loading: C:\projects\aqi_scad\02_Input\S5P_SO2_ugm3.zarr
  - RAW data range: -3793.31 to 3600.45
  - Time range: 2024-01-01T08:06:33.000000000 to 2024-12-30T22:55:03.000000000
  - Valid positive values: 30969768
   Loaded 8081 timesteps

2. Resampling to 1H using mean...
   Resampled to 8782 timesteps

3. Clipping to region...

4. Exporting TIFFs...
   Exported 8782 TIFF files

✓ SO2 processing complete!

Processing: NO2

1. Loading NO2 data...
Loading: C:\projects\aqi_scad\02_Input\CAMS_NO2_ugm3.zarr
  - RAW dat

KeyboardInterrupt: 

## Summary

In [7]:
# Print summary
print("\n" + "="*70)
print("PROCESSING SUMMARY")
print("="*70)
print(f"\nDate Range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"Sample Period: {CONFIG['sample_period']}")
print(f"Statistical Method: {CONFIG['stat_method']}")
print(f"Region: {CONFIG['region']}")
print(f"\nPollutants Processed: {', '.join(CONFIG['pollutant_names'])}")
print(f"\nOutput Directory: {output_folder}")
print(f"Files per Pollutant: {len(identifiers)}")
print(f"Total Files: {len(CONFIG['pollutant_names']) * len(identifiers)}")
print("="*70)


PROCESSING SUMMARY

Date Range: 2024-01-01 to 2024-12-31
Sample Period: 1YE
Statistical Method: mean
Region: All

Pollutants Processed: SO2, NO2, CO, O3, PM2P5, PM10

Output Directory: C:\projects\aqi_scad\03_Output_Data\yearly_data
Files per Pollutant: 1
Total Files: 6
